<a href="https://colab.research.google.com/github/CodeCommander11/retos-codigo/blob/main/Prediccion_Mundial_IA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
from google.colab import drive

# 1. Conectar Drive
drive.mount('/content/drive')

# 2. Definir la ruta raíz de tu proyecto
base_path = '/content/drive/MyDrive/Prediccion_Mundial_IA'

# 3. Lista de carpetas necesarias
folders = [
    'data/raw',       # Datos originales sin tocar
    'data/processed', # Datos limpios y con nuevas variables
    'models',         # Aquí guardaremos nuestros 3 modelos (Clasificador + Regresores)
    'outputs'         # Gráficas y reportes
]

# 4. Crear carpetas
for folder in folders:
    path = os.path.join(base_path, folder)
    os.makedirs(path, exist_ok=True)
    print(f"✅ Carpeta creada: {path}")

print("\n🚀 Estructura lista. Ahora todo lo que guardemos quedará respaldado en tu Drive.")

Mounted at /content/drive
✅ Carpeta creada: /content/drive/MyDrive/Prediccion_Mundial_IA/data/raw
✅ Carpeta creada: /content/drive/MyDrive/Prediccion_Mundial_IA/data/processed
✅ Carpeta creada: /content/drive/MyDrive/Prediccion_Mundial_IA/models
✅ Carpeta creada: /content/drive/MyDrive/Prediccion_Mundial_IA/outputs

🚀 Estructura lista. Ahora todo lo que guardemos quedará respaldado en tu Drive.


In [2]:
import os
import pandas as pd

# 1. Definir la nueva ruta base corregida
base_path = '/content/drive/MyDrive/Data_Science_Projects/Prediccion_Mundial_IA'

# 2. Asegurar que las carpetas existan en la nueva ubicación
folders = ['data/raw', 'data/processed', 'models', 'outputs']
for folder in folders:
    path = os.path.join(base_path, folder)
    os.makedirs(path, exist_ok=True)

# 3. Definir las rutas de archivos
path_raw = os.path.join(base_path, 'data/raw/resultados_historicos.csv')
path_processed = os.path.join(base_path, 'data/processed/partidos_limpios.csv')

# 4. Descargar y guardar
raw_data_url = "https://raw.githubusercontent.com/martj42/international_results/master/results.csv"

print("📥 Descargando datos originales...")
df_original = pd.read_csv(raw_data_url)

# Guardar en raw
df_original.to_csv(path_raw, index=False)
print(f"✅ Copia original guardada en: {path_raw}")

# 5. Limpieza y guardado en processed
df_original['date'] = pd.to_datetime(df_original['date'])
df = df_original.dropna(subset=['home_score', 'away_score']).copy()
df.to_csv(path_processed, index=False)
print(f"✅ Datos limpios guardados en: {path_processed}")

📥 Descargando datos originales...
✅ Copia original guardada en: /content/drive/MyDrive/Data_Science_Projects/Prediccion_Mundial_IA/data/raw/resultados_historicos.csv
✅ Datos limpios guardados en: /content/drive/MyDrive/Data_Science_Projects/Prediccion_Mundial_IA/data/processed/partidos_limpios.csv


In [3]:
# Verifica cuántos registros quedaron tras la limpieza
total_partidos = df.shape[0]
print(f"✅ Total de registros cargados y limpios: {total_partidos:,}")

# Vamos a ver los primeros años vs los últimos
print("\n📅 Rango de fechas:")
print(f"Desde: {df['date'].min()}")
print(f"Hasta: {df['date'].max()}")

✅ Total de registros cargados y limpios: 49,257

📅 Rango de fechas:
Desde: 1872-11-30 00:00:00
Hasta: 2026-03-31 00:00:00


In [4]:
# Creamos variables para medir el rendimiento reciente (últimos 5 partidos)
# Esto es vital para saber si un equipo está en racha

def calcular_rachas(df, ventana=5):
    # Ordenamos por fecha
    df = df.sort_values('date')

    # Calculamos el promedio de goles anotados y recibidos para cada equipo (Local y Visitante)
    # Nota: Esto es un ejemplo simplificado de la ingeniería necesaria
    df['home_form'] = df.groupby('home_team')['home_score'].transform(lambda x: x.rolling(ventana, min_periods=1).mean().shift(1))
    df['away_form'] = df.groupby('away_team')['away_score'].transform(lambda x: x.rolling(ventana, min_periods=1).mean().shift(1))

    return df

df = calcular_rachas(df)

# Guardamos el resultado procesado con estas nuevas columnas
df.to_csv(os.path.join(base_path, 'data/processed/partidos_con_features.csv'), index=False)

print("🚀 ¡Variables de forma calculadas y guardadas!")
display(df[['date', 'home_team', 'away_team', 'home_score', 'home_form', 'away_form']].tail())

🚀 ¡Variables de forma calculadas y guardadas!


,date,home_team,away_team,home_score,home_form,away_form
49229,2026-03-31,Botswana,Malawi,1.0,0.6,0.6
49230,2026-03-31,Mexico,Belgium,1.0,1.2,3.4
49231,2026-03-31,Montenegro,Slovenia,2.0,1.6,0.4
49234,2026-03-31,Niger,Togo,0.0,3.6,0.4
49256,2026-03-31,Kazakhstan,Comoros,1.0,1.4,0.4


In [5]:
# 1. Crear las etiquetas (Targets) para nuestros 3 motores
def resultado_match(row):
    if row['home_score'] > row['away_score']: return 2 # Gana Local
    elif row['home_score'] == row['away_score']: return 1 # Empate
    else: return 0 # Gana Visitante

df['target_resultado'] = df.apply(resultado_match, axis=1)
df['target_home_goals'] = df['home_score']
df['target_away_goals'] = df['away_score']

# 2. Split Temporal (evitamos trampa de futuro)
fecha_corte = '2024-01-01'
train = df[df['date'] < fecha_corte].copy()
test = df[df['date'] >= fecha_corte].copy()

# Guardar en la carpeta processed
train.to_csv(os.path.join(base_path, 'data/processed/train.csv'), index=False)
test.to_csv(os.path.join(base_path, 'data/processed/test.csv'), index=False)

print(f"✅ Entrenamiento: {train.shape[0]} partidos.")
print(f"✅ Prueba: {test.shape[0]} partidos.")
print("🚀 ¡Todo listo para entrenar los modelos!")

✅ Entrenamiento: 46859 partidos.
✅ Prueba: 2398 partidos.
🚀 ¡Todo listo para entrenar los modelos!


In [7]:
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder
import joblib
import pandas as pd
import os

# 1. Aseguramos que tenemos df (si no está cargado, lo cargamos)
base_path = '/content/drive/MyDrive/Data_Science_Projects/Prediccion_Mundial_IA'
df = pd.read_csv(os.path.join(base_path, 'data/processed/partidos_limpios.csv'))

# 2. FIT GLOBAL: Enseñamos todos los equipos posibles al Encoder
# Usamos .values.ravel() para aplanar las columnas de local y visitante y sacar todos los únicos
all_teams = pd.unique(df[['home_team', 'away_team']].values.ravel())
le = LabelEncoder()
le.fit(all_teams)

# 3. Aplicamos el encoder a los sets que ya teníamos divididos
train['home_team_id'] = le.transform(train['home_team'])
train['away_team_id'] = le.transform(train['away_team'])
test['home_team_id'] = le.transform(test['home_team'])
test['away_team_id'] = le.transform(test['away_team'])

# Features que usaremos
features = ['home_team_id', 'away_team_id', 'home_form', 'away_form']
X_train = train[features]
X_test = test[features]

print(f"✅ Encoder ajustado con {len(le.classes_)} equipos únicos.")
print("🚀 Entrenando los 3 motores de nuevo...")

# --- Re-Entrenamiento de los 3 modelos ---
modelo_res = xgb.XGBClassifier(n_estimators=300, learning_rate=0.05, device='cuda')
modelo_res.fit(X_train, train['target_resultado'])

modelo_home = xgb.XGBRegressor(n_estimators=300, learning_rate=0.05, device='cuda')
modelo_home.fit(X_train, train['target_home_goals'])

modelo_away = xgb.XGBRegressor(n_estimators=300, learning_rate=0.05, device='cuda')
modelo_away.fit(X_train, train['target_away_goals'])

# Guardado
joblib.dump(modelo_res, os.path.join(base_path, 'models/modelo_resultado.pkl'))
joblib.dump(modelo_home, os.path.join(base_path, 'models/modelo_goles_home.pkl'))
joblib.dump(modelo_away, os.path.join(base_path, 'models/modelo_goles_away.pkl'))
joblib.dump(le, os.path.join(base_path, 'models/label_encoder.pkl'))

print("✅ ¡Modelos re-entrenados y guardados con éxito!")

✅ Encoder ajustado con 336 equipos únicos.
🚀 Entrenando los 3 motores de nuevo...
✅ ¡Modelos re-entrenados y guardados con éxito!


In [12]:
import joblib
import pandas as pd
import numpy as np
import os

# 1. Definir rutas
base_path = '/content/drive/MyDrive/Data_Science_Projects/Prediccion_Mundial_IA'
# Cargamos los modelos
modelo_res = joblib.load(os.path.join(base_path, 'models/modelo_resultado.pkl'))
modelo_home = joblib.load(os.path.join(base_path, 'models/modelo_goles_home.pkl'))
modelo_away = joblib.load(os.path.join(base_path, 'models/modelo_goles_away.pkl'))
le = joblib.load(os.path.join(base_path, 'models/label_encoder.pkl'))

def predecir_partido(equipo_local, equipo_visitante):
    # CARGA SEGURA: Leemos los datos directamente del archivo para asegurar que 'home_form' exista
    df = pd.read_csv(os.path.join(base_path, 'data/processed/partidos_con_features.csv'))

    # Verificar si existen en el encoder
    if equipo_local not in le.classes_ or equipo_visitante not in le.classes_:
        return f"⚠️ Error: Uno de los equipos ('{equipo_local}' o '{equipo_visitante}') no está en la base de datos histórica."

    # Obtener IDs
    id_loc = le.transform([equipo_local])[0]
    id_vis = le.transform([equipo_visitante])[0]

    # Obtener el promedio de forma actual (usando los datos cargados)
    form_loc = df[df['home_team'] == equipo_local]['home_form'].mean()
    form_vis = df[df['away_team'] == equipo_visitante]['away_form'].mean()

    # Si el equipo no tiene historial, usamos 0.5 (promedio neutral)
    form_loc = form_loc if not np.isnan(form_loc) else 0.5
    form_vis = form_vis if not np.isnan(form_vis) else 0.5

    # Preparar input
    input_data = pd.DataFrame([[id_loc, id_vis, form_loc, form_vis]],
                              columns=['home_team_id', 'away_team_id', 'home_form', 'away_form'])

    # Predicciones
    prob_res = modelo_res.predict(input_data)[0]
    goles_loc = modelo_home.predict(input_data)[0]
    goles_vis = modelo_away.predict(input_data)[0]

    # Mapeo de resultados
    mapeo = {2: "Gana Local", 1: "Empate", 0: "Gana Visitante"}

    print(f"\n--- ⚽ Predicción: {equipo_local} vs {equipo_visitante} ---")
    print(f"Resultado probable: {mapeo[prob_res]}")
    print(f"Marcador estimado: {equipo_local} {round(goles_loc)} - {round(goles_vis)} {equipo_visitante}")
    print(f"(Goles esperados: {goles_loc:.2f} vs {goles_vis:.2f})")

# --- PRUEBA TU SIMULADOR ---
predecir_partido('Mexico', 'Belgium')
predecir_partido('Colombia', 'Argentina')


--- ⚽ Predicción: Mexico vs Belgium ---
Resultado probable: Gana Local
Marcador estimado: Mexico 2 - 1 Belgium
(Goles esperados: 1.77 vs 1.15)

--- ⚽ Predicción: Colombia vs Argentina ---
Resultado probable: Gana Visitante
Marcador estimado: Colombia 1 - 1 Argentina
(Goles esperados: 1.10 vs 1.36)


In [18]:
import joblib
from sklearn.metrics import classification_report, accuracy_score

# 1. Cargar el Encoder y los datos
le = joblib.load(os.path.join(base_path, 'models/label_encoder.pkl'))
test = pd.read_csv(os.path.join(base_path, 'data/processed/test.csv'))

# 2. Transformar los datos de prueba para que el modelo los entienda
# ¡Aquí está la solución al error!
test['home_team_id'] = le.transform(test['home_team'])
test['away_team_id'] = le.transform(test['away_team'])

# 3. Preparar X_test y y_true
X_test = test[['home_team_id', 'away_team_id', 'home_form', 'away_form']]
y_true = test['target_resultado']

# 4. Predecir
y_pred = modelo_res.predict(X_test)

# 5. Resultados
print("--- 📊 Reporte de Rendimiento del Modelo ---")
print(f"Precisión Global (Accuracy): {accuracy_score(y_true, y_pred)*100:.2f}%")
print("\nDesglose por categoría:")
print(classification_report(y_true, y_pred, target_names=['Gana Vis', 'Empate', 'Gana Loc']))


--- 📊 Reporte de Rendimiento del Modelo ---
Precisión Global (Accuracy): 54.05%

Desglose por categoría:
              precision    recall  f1-score   support

    Gana Vis       0.58      0.36      0.44       701
      Empate       0.33      0.01      0.02       571
    Gana Loc       0.53      0.92      0.68      1126

    accuracy                           0.54      2398
   macro avg       0.48      0.43      0.38      2398
weighted avg       0.50      0.54      0.45      2398



In [19]:
import joblib
import pandas as pd
import numpy as np
import os

# Aseguramos que los modelos estén cargados
base_path = '/content/drive/MyDrive/Data_Science_Projects/Prediccion_Mundial_IA'
modelo_res = joblib.load(os.path.join(base_path, 'models/modelo_resultado.pkl'))
modelo_home = joblib.load(os.path.join(base_path, 'models/modelo_goles_home.pkl'))
modelo_away = joblib.load(os.path.join(base_path, 'models/modelo_goles_away.pkl'))
le = joblib.load(os.path.join(base_path, 'models/label_encoder.pkl'))

def predecir_partido_con_confianza(equipo_local, equipo_visitante):
    # Cargar datos para obtener el 'form'
    df = pd.read_csv(os.path.join(base_path, 'data/processed/partidos_con_features.csv'))

    if equipo_local not in le.classes_ or equipo_visitante not in le.classes_:
        return "⚠️ Error: Equipo no encontrado."

    id_loc = le.transform([equipo_local])[0]
    id_vis = le.transform([equipo_visitante])[0]

    form_loc = df[df['home_team'] == equipo_local]['home_form'].mean()
    form_vis = df[df['away_team'] == equipo_visitante]['away_form'].mean()

    form_loc = form_loc if not np.isnan(form_loc) else 0.5
    form_vis = form_vis if not np.isnan(form_vis) else 0.5

    # DEFINICIÓN CORRECTA DE INPUT_DATA
    input_data = pd.DataFrame([[id_loc, id_vis, form_loc, form_vis]],
                              columns=['home_team_id', 'away_team_id', 'home_form', 'away_form'])

    # Probabilidades
    probs = modelo_res.predict_proba(input_data)[0]

    # Predicciones de goles
    goles_loc = modelo_home.predict(input_data)[0]
    goles_vis = modelo_away.predict(input_data)[0]

    # Mapeo
    mapeo = {0: "Gana Visitante", 1: "Empate", 2: "Gana Local"}
    pred_idx = np.argmax(probs)

    print(f"\n--- ⚽ Análisis: {equipo_local} vs {equipo_visitante} ---")
    print(f"Predicción final: {mapeo[pred_idx]}")
    print(f"Marcador estimado: {round(goles_loc)} - {round(goles_vis)}")
    print(f"Confianza del modelo:")
    print(f"  - Gana Visitante: {probs[0]*100:.1f}%")
    print(f"  - Empate: {probs[1]*100:.1f}%")
    print(f"  - Gana Local: {probs[2]*100:.1f}%")

# Ahora sí, llámala:
predecir_partido_con_confianza('Mexico', 'Belgium')
predecir_partido_con_confianza('Colombia', 'Argentina')


--- ⚽ Análisis: Mexico vs Belgium ---
Predicción final: Gana Local
Marcador estimado: 2 - 1
Confianza del modelo:
  - Gana Visitante: 23.4%
  - Empate: 26.7%
  - Gana Local: 49.9%

--- ⚽ Análisis: Colombia vs Argentina ---
Predicción final: Gana Visitante
Marcador estimado: 1 - 1
Confianza del modelo:
  - Gana Visitante: 43.4%
  - Empate: 30.7%
  - Gana Local: 25.9%


In [20]:
# Definir parámetros de Elo
# K es la sensibilidad: un valor de 20 es estándar para fútbol internacional
K = 20

def calcular_elo(df):
    # Diccionario para guardar el elo actual de cada equipo
    elo_dict = {}

    # Inicializamos todos los equipos con 1500 puntos
    equipos = pd.unique(df[['home_team', 'away_team']].values.ravel())
    for equipo in equipos:
        elo_dict[equipo] = 1500.0

    df = df.sort_values('date')
    elo_hist_home = []
    elo_hist_away = []

    for i, row in df.iterrows():
        # Obtenemos Elo actual
        elo_h = elo_dict[row['home_team']]
        elo_a = elo_dict[row['away_team']]

        # Guardamos en la lista
        elo_hist_home.append(elo_h)
        elo_hist_away.append(elo_a)

        # Calculamos la probabilidad esperada
        expectativa_h = 1 / (1 + 10 ** ((elo_a - elo_h) / 400))

        # Resultado real (1 si gana local, 0.5 empate, 0 si pierde)
        if row['home_score'] > row['away_score']:
            resultado = 1.0
        elif row['home_score'] == row['away_score']:
            resultado = 0.5
        else:
            resultado = 0.0

        # Actualizamos Elo
        elo_dict[row['home_team']] += K * (resultado - expectativa_h)
        elo_dict[row['away_team']] += K * ((1 - resultado) - (1 - expectativa_h))

    return elo_hist_home, elo_hist_away

# Aplicamos la función
df['elo_home'], df['elo_away'] = calcular_elo(df)

# Guardamos el archivo actualizado
df.to_csv(os.path.join(base_path, 'data/processed/partidos_con_elo.csv'), index=False)

print("✅ Sistema Elo implementado y guardado en 'partidos_con_elo.csv'.")
print("🚀 Ahora el modelo sabe la 'fuerza' real de cada país.")

✅ Sistema Elo implementado y guardado en 'partidos_con_elo.csv'.
🚀 Ahora el modelo sabe la 'fuerza' real de cada país.


In [21]:
import numpy as np
from scipy.stats import poisson

def calcular_probabilidades_poisson(goles_loc_esperados, goles_vis_esperados, max_goles=7):
    """
    Calcula la probabilidad de victoria/empate/derrota basada en Poisson.
    """
    # Creamos una matriz de probabilidades de 0 a 6 goles para cada equipo
    probs_home = [poisson.pmf(i, goles_loc_esperados) for i in range(max_goles)]
    probs_away = [poisson.pmf(i, goles_vis_esperados) for i in range(max_goles)]

    # Creamos la matriz de resultados (producto exterior)
    # Fila = Goles Local, Columna = Goles Visitante
    matriz_prob = np.outer(probs_home, probs_away)

    # Sumar probabilidades
    prob_gana_local = np.sum(np.tril(matriz_prob, -1)) # Suma abajo de la diagonal
    prob_empate = np.sum(np.diag(matriz_prob))          # Suma de la diagonal
    prob_gana_vis = np.sum(np.triu(matriz_prob, 1))    # Suma arriba de la diagonal

    # Normalizar por si acaso (la suma debe ser 1)
    total = prob_gana_local + prob_empate + prob_gana_vis
    return prob_gana_local/total, prob_empate/total, prob_gana_vis/total

# Prueba rápida con valores hipotéticos
p_local, p_empate, p_vis = calcular_probabilidades_poisson(1.8, 1.2)

print(f"--- 🧮 Análisis Matemático (Poisson) ---")
print(f"Prob. Victoria Local: {p_local*100:.2f}%")
print(f"Prob. Empate: {p_empate*100:.2f}%")
print(f"Prob. Victoria Visitante: {p_vis*100:.2f}%")

--- 🧮 Análisis Matemático (Poisson) ---
Prob. Victoria Local: 51.33%
Prob. Empate: 23.20%
Prob. Victoria Visitante: 25.47%


In [27]:
import joblib
import pandas as pd
import xgboost as xgb
import os
import numpy as np

base_path = '/content/drive/MyDrive/Data_Science_Projects/Prediccion_Mundial_IA'

# 1. Cargar el dataset
df = pd.read_csv(os.path.join(base_path, 'data/processed/partidos_con_elo.csv'))

# 2. AUTO-REPARACIÓN: Si faltan los targets, los creamos al instante
print(f"Columnas disponibles: {df.columns.tolist()}")

if 'target_resultado' not in df.columns:
    print("⚠️ Faltan targets. Calculando targets...")
    # Crear target_resultado (2: Gana Local, 1: Empate, 0: Gana Vis)
    df['target_resultado'] = np.where(df['home_score'] > df['away_score'], 2,
                                      np.where(df['home_score'] == df['away_score'], 1, 0))
    # Crear targets de goles
    df['target_home_goals'] = df['home_score']
    df['target_away_goals'] = df['away_score']

# 3. Asegurar Formas (Calculamos si no existen)
if 'home_form' not in df.columns:
    df = df.sort_values('date')
    df['home_form'] = df.groupby('home_team')['home_score'].transform(lambda x: x.rolling(window=5, min_periods=1).mean())
    df['away_form'] = df.groupby('away_team')['away_score'].transform(lambda x: x.rolling(window=5, min_periods=1).mean())
    df = df.fillna(0.5)

# 4. Asegurar IDs
le = joblib.load(os.path.join(base_path, 'models/label_encoder.pkl'))
df['home_team_id'] = le.transform(df['home_team'])
df['away_team_id'] = le.transform(df['away_team'])

# 5. Split y Entrenamiento
fecha_corte = '2024-01-01'
train = df[df['date'] < fecha_corte].copy()

features = ['home_team_id', 'away_team_id', 'home_form', 'away_form', 'elo_home', 'elo_away']
X_train = train[features]

print("🚀 Entrenando modelos...")
modelo_res = xgb.XGBClassifier(n_estimators=500, learning_rate=0.05, device='cuda')
modelo_res.fit(X_train, train['target_resultado'])

modelo_home = xgb.XGBRegressor(n_estimators=500, learning_rate=0.05, device='cuda')
modelo_home.fit(X_train, train['target_home_goals'])

modelo_away = xgb.XGBRegressor(n_estimators=500, learning_rate=0.05, device='cuda')
modelo_away.fit(X_train, train['target_away_goals'])

# Guardar modelos
joblib.dump(modelo_res, os.path.join(base_path, 'models/modelo_resultado.pkl'))
joblib.dump(modelo_home, os.path.join(base_path, 'models/modelo_goles_home.pkl'))
joblib.dump(modelo_away, os.path.join(base_path, 'models/modelo_goles_away.pkl'))

print("✅ ¡Modelos re-entrenados con éxito!")

Columnas disponibles: ['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'city', 'country', 'neutral', 'elo_home', 'elo_away']
⚠️ Faltan targets. Calculando targets...
🚀 Entrenando modelos...
✅ ¡Modelos re-entrenados con éxito!


In [28]:
import numpy as np
import pandas as pd
import joblib
import os
from scipy.stats import poisson

# Aseguramos cargar lo necesario
le = joblib.load(os.path.join(base_path, 'models/label_encoder.pkl'))
modelo_res = joblib.load(os.path.join(base_path, 'models/modelo_resultado.pkl'))
modelo_home = joblib.load(os.path.join(base_path, 'models/modelo_goles_home.pkl'))
modelo_away = joblib.load(os.path.join(base_path, 'models/modelo_goles_away.pkl'))

def predictor_unificado(equipo_local, equipo_visitante):
    # 1. Buscar los datos más recientes de estos equipos en el dataframe
    try:
        # Buscamos la última fila disponible para cada equipo para obtener su estado actual
        info_loc = df[df['home_team'] == equipo_local].iloc[-1]
        info_vis = df[df['away_team'] == equipo_visitante].iloc[-1]

        # Extraer variables
        id_loc = le.transform([equipo_local])[0]
        id_vis = le.transform([equipo_visitante])[0]
        elo_loc = info_loc['elo_home']
        elo_vis = info_vis['elo_away']
        form_loc = info_loc['home_form']
        form_vis = info_vis['away_form']
    except Exception as e:
        return f"Error al buscar datos: {e}. ¿Estás seguro que estos equipos existen en el dataset?"

    # 2. Crear input_data (debe tener las mismas columnas que el entrenamiento)
    input_data = pd.DataFrame([[id_loc, id_vis, form_loc, form_vis, elo_loc, elo_vis]],
                              columns=['home_team_id', 'away_team_id', 'home_form', 'away_form', 'elo_home', 'elo_away'])

    # 3. XGBoost: Probabilidad Clasificadora [Vis, Emp, Loc]
    probs_xgb = modelo_res.predict_proba(input_data)[0]

    # 4. Poisson: Probabilidad de Goles
    goles_h = modelo_home.predict(input_data)[0]
    goles_a = modelo_away.predict(input_data)[0]

    # 5. Cálculo Poisson
    # Matriz de probabilidad (0 a 6 goles)
    max_g = 7
    probs_h = [poisson.pmf(i, goles_h) for i in range(max_g)]
    probs_a = [poisson.pmf(i, goles_a) for i in range(max_g)]
    matriz = np.outer(probs_h, probs_a)

    p_loc = np.sum(np.tril(matriz, -1)) # Gana Local
    p_emp = np.sum(np.diag(matriz))      # Empate
    p_vis = np.sum(np.triu(matriz, 1))    # Gana Vis
    total = p_loc + p_emp + p_vis

    # Normalizar
    p_loc, p_emp, p_vis = p_loc/total, p_emp/total, p_vis/total

    # 6. CONSENSO (Promedio simple)
    final_vis = (probs_xgb[0] + p_vis) / 2
    final_emp = (probs_xgb[1] + p_emp) / 2
    final_loc = (probs_xgb[2] + p_loc) / 2

    mapeo = ["Gana Visitante", "Empate", "Gana Local"]
    ganador = mapeo[np.argmax([final_vis, final_emp, final_loc])]

    print(f"\n--- 🎯 Consenso Final: {equipo_local} vs {equipo_visitante} ---")
    print(f"Predicción: {ganador}")
    print(f"Probabilidades: Local: {final_loc*100:.1f}% | Empate: {final_emp*100:.1f}% | Visitante: {final_vis*100:.1f}%")
    print(f"Goles estimados: {goles_h:.1f} - {goles_a:.1f}")

# Probar ahora:
predictor_unificado('Mexico', 'Belgium')
predictor_unificado('Colombia', 'Argentina')


--- 🎯 Consenso Final: Mexico vs Belgium ---
Predicción: Gana Visitante
Probabilidades: Local: 6.2% | Empate: 14.0% | Visitante: 79.8%
Goles estimados: 0.8 - 3.1

--- 🎯 Consenso Final: Colombia vs Argentina ---
Predicción: Gana Visitante
Probabilidades: Local: 19.2% | Empate: 33.7% | Visitante: 47.0%
Goles estimados: 1.6 - 2.4


In [30]:
import pandas as pd

# 1. Crear una clave única para el enfrentamiento (sin importar el orden local/visita)
def get_matchup_id(row):
    teams = sorted([row['home_team'], row['away_team']])
    return f"{teams[0]}_vs_{teams[1]}"

print("🚀 Calculando H2H de forma optimizada...")

# Creamos la columna matchup para agrupar
df['matchup'] = df.apply(get_matchup_id, axis=1)

# 2. Definir quién ganó (1 si ganó el local, 0 si no)
# Nota: Esto es una simplificación de "ventaja del equipo que juega en casa"
df['won'] = (df['home_score'] > df['away_score']).astype(int)

# 3. Calcular la racha histórica acumulada usando groupby
# Esto es miles de veces más rápido que un bucle
df['h2h_win_rate'] = df.groupby(['matchup', 'home_team'])['won'] \
                       .transform(lambda x: x.shift().expanding().mean().fillna(0.5))

# 4. Limpieza: eliminamos columnas auxiliares
df = df.drop(columns=['matchup', 'won'])

# Guardamos
df.to_csv(os.path.join(base_path, 'data/processed/partidos_con_h2h.csv'), index=False)
print("✅ H2H implementado correctamente y base de datos guardada.")

🚀 Calculando H2H de forma optimizada...
✅ H2H implementado correctamente y base de datos guardada.


In [32]:
import pandas as pd
import joblib
import os
from sklearn.calibration import CalibratedClassifierCV

base_path = '/content/drive/MyDrive/Data_Science_Projects/Prediccion_Mundial_IA'

# 1. Cargar datos frescos
df = pd.read_csv(os.path.join(base_path, 'data/processed/partidos_con_h2h.csv'))
le = joblib.load(os.path.join(base_path, 'models/label_encoder.pkl'))

# 2. Re-aplicar procesamiento (para asegurar que 'test' tenga las columnas)
df['home_team_id'] = le.transform(df['home_team'])
df['away_team_id'] = le.transform(df['away_team'])

# Asegurar formas y targets (por si acaso)
df['home_form'] = df.groupby('home_team')['home_score'].transform(lambda x: x.rolling(window=5, min_periods=1).mean()).fillna(0.5)
df['away_form'] = df.groupby('away_team')['away_score'].transform(lambda x: x.rolling(window=5, min_periods=1).mean()).fillna(0.5)
df['target_resultado'] = (df['home_score'] > df['away_score']).astype(int) # Simplificación para el clasificador

# 3. Definir features y Split
features = ['home_team_id', 'away_team_id', 'home_form', 'away_form', 'elo_home', 'elo_away']
fecha_corte = '2024-01-01'
test = df[df['date'] >= fecha_corte].copy()

X_test = test[features]
y_test = test['target_resultado']

# 4. Calibración
print("🔄 Cargando modelo base y calibrando...")
modelo_res = joblib.load(os.path.join(base_path, 'models/modelo_resultado.pkl'))
calibrated_clf = CalibratedClassifierCV(estimator=modelo_res, method='sigmoid', cv='prefit')
calibrated_clf.fit(X_test, y_test)

# 5. Guardar
joblib.dump(calibrated_clf, os.path.join(base_path, 'models/modelo_resultado_calibrado.pkl'))

print("✅ ¡Modelo calibrado y guardado como 'modelo_resultado_calibrado.pkl'!")

🔄 Cargando modelo base y calibrando...
✅ ¡Modelo calibrado y guardado como 'modelo_resultado_calibrado.pkl'!


/usr/local/lib/python3.12/dist-packages/sklearn/calibration.py:333: UserWarning: The `cv='prefit'` option is deprecated in 1.6 and will be removed in 1.8. You can use CalibratedClassifierCV(FrozenEstimator(estimator)) instead.
  warnings.warn(


In [36]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 9.9 MB/s eta 0:00:00


In [50]:
# Asegurar que todas las librerías estén cargadas
import optuna
import xgboost as xgb
import joblib
import pandas as pd
import os
import numpy as np
from sklearn.metrics import accuracy_score

# Configuración de entorno
base_path = '/content/drive/MyDrive/Data_Science_Projects/Prediccion_Mundial_IA'

# Carga y Preparación de datos (Blindado contra errores)
df = pd.read_csv(os.path.join(base_path, 'data/processed/partidos_con_h2h.csv'))
le = joblib.load(os.path.join(base_path, 'models/label_encoder.pkl'))

# Aplicar transformaciones necesarias
df['home_team_id'] = le.transform(df['home_team'])
df['away_team_id'] = le.transform(df['away_team'])
df['target_resultado'] = np.where(df['home_score'] > df['away_score'], 2, np.where(df['home_score'] == df['away_score'], 1, 0))

# Definir features (incluyendo h2h_win_rate que ya creamos antes)
features = ['home_team_id', 'away_team_id', 'home_form', 'away_form', 'elo_home', 'elo_away', 'h2h_win_rate']
fecha_corte = '2024-01-01'

train = df[df['date'] < fecha_corte].copy()
test = df[df['date'] >= fecha_corte].copy()

X_train, X_test = train[features], test[features]
y_train, y_test = train['target_resultado'], test['target_resultado']

# Función de optimización
def objective(trial):
    param = {
        'n_estimators': trial.suggest_int('n_estimators', 300, 1000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 9),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'device': 'cuda'
    }

    # Entrenar modelo temporal
    clf = xgb.XGBClassifier(**param, use_label_encoder=False, eval_metric='mlogloss')
    clf.fit(X_train, y_train)

    # Evaluar
    preds = clf.predict(X_test)
    return accuracy_score(y_test, preds)

# Ejecutar optimización
print("🧠 Iniciando Optuna: Buscando la configuración perfecta...")
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20)

print(f"✅ ¡Optimización finalizada!")
print(f"Mejor Accuracy: {study.best_value:.4f}")
print(f"Mejores parámetros: {study.best_params}")

# Guardar modelo final con los mejores parámetros
best_model = xgb.XGBClassifier(**study.best_params, device='cuda')
best_model.fit(X_train, y_train)
joblib.dump(best_model, os.path.join(base_path, 'models/modelo_resultado_optimizado.pkl'))

print("💾 Modelo optimizado guardado exitosamente.")

[I 2026-05-22 01:18:27,802] A new study created in memory with name: no-name-f412f5c2-b9f1-41ec-ba6c-3cbd70e04bec


🧠 Iniciando Optuna: Buscando la configuración perfecta...


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [01:18:27] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
[I 2026-05-22 01:18:30,589] Trial 0 finished with value: 0.6163469557964971 and parameters: {'n_estimators': 730, 'learning_rate': 0.02707626151666387, 'max_depth': 5, 'subsample': 0.9084872231943109, 'colsample_bytree': 0.968734105599601}. Best is trial 0 with value: 0.6163469557964971.
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [01:18:30] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
[I 2026-05-22 01:18:34,377] Trial 1 finished with value: 0.6075896580483736 and parameters: {'n_estimators': 472, 'learning_rate': 0.02579284072244505, 'max_depth': 8, 'subsample': 0.8897782387152052, 'colsample_bytree': 0.7031712681814425}.

✅ ¡Optimización finalizada!
Mejor Accuracy: 0.6209
Mejores parámetros: {'n_estimators': 617, 'learning_rate': 0.051860649559781584, 'max_depth': 3, 'subsample': 0.6035081421013627, 'colsample_bytree': 0.9969484847958338}
💾 Modelo optimizado guardado exitosamente.


In [40]:
def reporte_prediccion(home_team, away_team, banca=100, cuota_apuesta=2.0):
    try:
        # 1. Cargar datos y modelo
        modelo_opt = joblib.load(os.path.join(base_path, 'models/modelo_resultado_optimizado.pkl'))

        # Asegurar que tenemos los datos más recientes
        df = pd.read_csv(os.path.join(base_path, 'data/processed/partidos_con_h2h.csv'))

        # 2. Obtener datos
        info_loc = df[df['home_team'] == home_team].iloc[-1]
        info_vis = df[df['away_team'] == away_team].iloc[-1]

        # 3. Definir las columnas EXACTAS (Orden crucial)
        cols = ['home_team_id', 'away_team_id', 'home_form', 'away_form', 'elo_home', 'elo_away', 'h2h_win_rate']

        # 4. Crear el DataFrame forzando las columnas
        data = [[
            le.transform([home_team])[0],
            le.transform([away_team])[0],
            info_loc['home_form'],
            info_vis['away_form'],
            info_loc['elo_home'],
            info_vis['elo_away'],
            info_loc['h2h_win_rate']
        ]]

        input_data = pd.DataFrame(data, columns=cols)

        # VERIFICACIÓN DE SEGURIDAD
        print(f"DEBUG: Columnas enviadas al modelo: {list(input_data.columns)}")

        # 5. Predicción
        probs = modelo_opt.predict_proba(input_data)[0]

        # 6. Cálculo de Kelly
        prob_win = probs[2]
        b = cuota_apuesta - 1
        kelly = (prob_win * b - (1 - prob_win)) / b
        recomendacion = max(0, kelly * 0.25) * 100

        print(f"\n--- 📊 REPORTE DE APUESTAS ---")
        print(f"{home_team} vs {away_team}")
        print(f"Probabilidad de Victoria Local: {probs[2]*100:.1f}%")
        print(f"--------------------------------------------------")
        print(f"💰 Gestión de Banca: {recomendacion:.2f}% de tu banca.")
        print(f"--------------------------------------------------\n")

    except Exception as e:
        print(f"❌ ERROR CRÍTICO: {e}")
        # Si esto falla, es porque el modelo guardado tiene menos columnas de las que creemos.
        # Imprimiremos las columnas que el modelo espera:
        print(f"Features del modelo: {modelo_opt.feature_names_in_ if hasattr(modelo_opt, 'feature_names_in_') else 'No disponibles'}")

# Probar
reporte_prediccion("Mexico", "Belgium", banca=1000, cuota_apuesta=2.10)

DEBUG: Columnas enviadas al modelo: ['home_team_id', 'away_team_id', 'home_form', 'away_form', 'elo_home', 'elo_away', 'h2h_win_rate']

--- 📊 REPORTE DE APUESTAS ---
Mexico vs Belgium
Probabilidad de Victoria Local: 21.0%
--------------------------------------------------
💰 Gestión de Banca: 0.00% de tu banca.
--------------------------------------------------



In [43]:
def reporte_prediccion(home_team, away_team, banca=100, cuota_apuesta=2.0):
    try:
        # 1. Cargar modelo y datos
        modelo_opt = joblib.load(os.path.join(base_path, 'models/modelo_resultado_optimizado.pkl'))
        df = pd.read_csv(os.path.join(base_path, 'data/processed/partidos_con_h2h.csv'))

        # 2. Validación de nombres (¡Aquí evitamos el error!)
        teams_in_db = pd.concat([df['home_team'], df['away_team']]).unique()

        if home_team not in teams_in_db or away_team not in teams_in_db:
            print(f"❌ ERROR: Uno de los equipos no fue encontrado.")
            print(f"   Tu búsqueda: {home_team} vs {away_team}")
            print(f"   Equipos disponibles (ejemplo): {teams_in_db[:10]}...")
            return

        # 3. Obtener datos (usando .loc para filtrar)
        info_loc = df[df['home_team'] == home_team].iloc[-1]
        info_vis = df[df['away_team'] == away_team].iloc[-1]

        # 4. Crear DataFrame
        cols = ['home_team_id', 'away_team_id', 'home_form', 'away_form', 'elo_home', 'elo_away', 'h2h_win_rate']
        input_data = pd.DataFrame([[
            le.transform([home_team])[0],
            le.transform([away_team])[0],
            info_loc['home_form'],
            info_vis['away_form'],
            info_loc['elo_home'],
            info_vis['elo_away'],
            info_loc['h2h_win_rate']
        ]], columns=cols)

        # 5. Predicción
        probs = modelo_opt.predict_proba(input_data)[0]
        prob_win = probs[2]

        # 6. Reporte
        print(f"\n--- 📊 REPORTE: {home_team} vs {away_team} ---")
        print(f"Probabilidad de Victoria Local: {prob_win*100:.1f}%")

    except Exception as e:
        print(f"❌ ERROR CRÍTICO: {e}")

# Prueba con un equipo que SEPAS que está en tu CSV:
# reporte_prediccion("Mexico", "Belgium", banca=1000, cuota_apuesta=2.10)

In [45]:
# Ver nombres exactos de los primeros 20 equipos
print(df['home_team'].unique()[:20])

['Scotland' 'England' 'Wales' 'Northern Ireland' 'United States' 'Uruguay'
 'Austria' 'Hungary' 'Argentina' 'Belgium' 'France' 'Guernsey' 'Jersey'
 'Netherlands' 'Guyana' 'Czechoslovakia' 'Alderney' 'Switzerland' 'Sweden'
 'Germany']


In [46]:
# Ambos equipos están en tu lista, así que esto debería funcionar sin errores:
reporte_prediccion("Argentina", "Belgium", banca=1000, cuota_apuesta=2.10)


--- 📊 REPORTE: Argentina vs Belgium ---
Probabilidad de Victoria Local: 31.7%


In [53]:
def reporte_prediccion(home_team, away_team, banca=100, cuota_apuesta=2.0):
    try:
        # Cargar modelo
        modelo_opt = joblib.load(os.path.join(base_path, 'models/modelo_resultado_optimizado.pkl'))

        # Obtener datos
        info_loc = df[df['home_team'] == home_team].iloc[-1]
        info_vis = df[df['away_team'] == away_team].iloc[-1]

        # Preparar entrada
        input_data = pd.DataFrame([[
            le.transform([home_team])[0], le.transform([away_team])[0],
            info_loc['home_form'], info_vis['away_form'],
            info_loc['elo_home'], info_vis['elo_away'], info_loc['h2h_win_rate']
        ]], columns=['home_team_id', 'away_team_id', 'home_form', 'away_form', 'elo_home', 'elo_away', 'h2h_win_rate'])

        # Predicciones
        probs = modelo_opt.predict_proba(input_data)[0]
        prob_win = probs[2]

        # Cálculo de recomendación (Kelly al 25%)
        b = cuota_apuesta - 1
        kelly = (prob_win * b - (1 - prob_win)) / b
        recomendacion = max(0, kelly * 0.25) * 100

        # Output limpio
        print(f"\n--- 📊 REPORTE DE PREDICCIÓN ---")
        print(f"Partido: {home_team} vs {away_team}")
        print(f"Probabilidad de Victoria Local: {prob_win*100:.1f}%")
        print(f"💰 Recomendación: {recomendacion:.2f}% de tu banca.")
        print("----------------------------------\n")

    except Exception as e:
        print(f"❌ Ocurrió un error al procesar el reporte: {e}")

# Ejecutar interfaz
display(widgets.VBox([dropdown_home, dropdown_away, btn_calcular, output]))